In [ ]:
import os, shutil, pathlib

# Match actual folder name: breed/Images_final (capital I)
original_dir = pathlib.Path("breed/Images_final")

In [ ]:
import os, shutil, pathlib


new_base_dir = pathlib.Path("dog_breed")

import os, random, shutil
from pathlib import Path

def make_subset(original_dir, new_base_dir, subset_name, split_map, seed=123):
    """
    original_dir:
        dogs/
          Beagle/*.jpg
          Husky/*.jpg
    new_base_dir:
        output/
          train/Beagle/*.jpg ...
          validation/Beagle/*.jpg ...
          test/Beagle/*.jpg ...
    split_map example:
        {"train": (0.0, 0.7), "validation": (0.7, 0.9), "test": (0.9, 1.0)}
    """
    random.seed(seed)

    source_dir = Path(original_dir)
    new_base_dir = Path(new_base_dir)

    # each subfolder is a class (breed)
    breed_dirs = [d for d in source_dir.iterdir() if d.is_dir()]

    for breed_dir in breed_dirs:
        breed = breed_dir.name

        images = [p for p in breed_dir.iterdir() if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]]
        images.sort()  # deterministic before shuffle
        random.shuffle(images)

        n = len(images)
        start_frac, end_frac = split_map[subset_name]
        start_idx = int(n * start_frac)
        end_idx = int(n * end_frac)

        target_dir = new_base_dir / subset_name / breed
        os.makedirs(target_dir, exist_ok=True)

        for src_path in images[start_idx:end_idx]:
            dst_path = target_dir / src_path.name
            shutil.copy2(src_path, dst_path)

# ---- usage ----

split_map = {
    "train": (0.0, 0.7),
    "validation": (0.7, 0.9),
    "test": (0.9, 1.0),
}

make_subset(original_dir, new_base_dir, "train", split_map, seed=123)
make_subset(original_dir, new_base_dir, "validation", split_map, seed=123)
make_subset(original_dir, new_base_dir, "test", split_map, seed=123)

print("Done!")


In [ ]:
import os
from pathlib import Path
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import hashlib
import random
import numpy as np

DATASET_DIR = Path("dog_breed")  # change if needed
SPLITS = ["train", "validation", "test"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def iter_images(split_dir: Path):
    for breed_dir in split_dir.iterdir():
        if breed_dir.is_dir():
            breed = breed_dir.name
            for p in breed_dir.rglob("*"):
                if p.suffix.lower() in IMG_EXTS:
                    yield breed, p

def file_md5(path: Path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

rows = []
corrupt = []

for split in SPLITS:
    split_dir = DATASET_DIR / split
    if not split_dir.exists():
        print(f"Missing folder: {split_dir}")
        continue

    for breed, img_path in iter_images(split_dir):
        try:
            with Image.open(img_path) as im:
                im.verify()  # checks corruption
            # reopen after verify to read size/mode
            with Image.open(img_path) as im:
                w, h = im.size
                mode = im.mode
            rows.append({
                "split": split,
                "breed": breed,
                "path": str(img_path),
                "width": w,
                "height": h,
                "aspect_ratio": w / h if h else None,
                "mode": mode,
                "filesize_kb": img_path.stat().st_size / 1024.0
            })
        except Exception as e:
            corrupt.append({"split": split, "breed": breed, "path": str(img_path), "error": str(e)})

df = pd.DataFrame(rows)
corrupt_df = pd.DataFrame(corrupt)

print("=== Basic summary ===")
print("Total images:", len(df))
print("Breeds:", df["breed"].nunique())
print("\nImages per split:")
print(df["split"].value_counts())

if len(corrupt_df) > 0:
    print("\n=== Corrupt/unreadable images ===")
    print(corrupt_df.head(20))
else:
    print("\nNo corrupt images found ✅")

# 1) Class counts


In [ ]:
import pandas as pd

counts = df.groupby(["split", "breed"]).size().reset_index(name="count")

for split in ["train", "validation", "test"]:
    s = counts[counts["split"] == split].sort_values("count", ascending=False)
    print(f"\n--- {split.upper()} ---")
    print("Top 10:\n", s.head(10).to_string(index=False))
    print("Bottom 10:\n", s.tail(10).to_string(index=False))
    print("Min/Max:", s["count"].min(), "/", s["count"].max())


In [ ]:
splits = ["train","validation","test"]
breed_sets = {s: set(df[df["split"]==s]["breed"].unique()) for s in splits}

print("Missing in validation:", sorted(breed_sets["train"] - breed_sets["validation"])[:20])
print("Missing in test:", sorted(breed_sets["train"] - breed_sets["test"])[:20])

print("Breeds only in validation:", sorted(breed_sets["validation"] - breed_sets["train"])[:20])
print("Breeds only in test:", sorted(breed_sets["test"] - breed_sets["train"])[:20])


In [ ]:
print(df[["width","height","aspect_ratio","filesize_kb"]].describe())

# quick flags
small = df[(df["width"] < 80) | (df["height"] < 80)]
print("Very small images:", len(small))

weird_ar = df[(df["aspect_ratio"] > 3.0) | (df["aspect_ratio"] < 0.33)]
print("Very weird aspect ratios:", len(weird_ar))


In [ ]:
weird = df[(df["aspect_ratio"] > 3.0) | (df["aspect_ratio"] < 0.33)]
print(weird[["split","breed","width","height","aspect_ratio","path"]])


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

p = weird.iloc[0]["path"]
img = Image.open(p).convert("RGB")
plt.figure(figsize=(6,6))
plt.imshow(img)
plt.title(p)
plt.axis("off")
plt.show()


In [ ]:
from keras.utils import image_dataset_from_directory

batch_size = 64
image_size = (160, 160)
train_dataset = image_dataset_from_directory(
    new_base_dir / "train", image_size=image_size, batch_size=batch_size
)
validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation", image_size=image_size, batch_size=batch_size
)
test_dataset = image_dataset_from_directory(
    new_base_dir / "test", image_size=image_size, batch_size=batch_size
)

In [ ]:
import keras
from keras import layers

num_classes = 120
img_size = (160, 160)  # try 160; if slow, use (128,128)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])

base = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=img_size + (3,)
)
base.trainable = False  # freeze first

inputs = keras.Input(shape=img_size + (3,))
x = data_augmentation(inputs)
x = keras.applications.efficientnet.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
model.summary(line_length=80)


In [ ]:
for data_batch, labels_batch in train_dataset:
    print("data batch shape:", data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    break

In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


In [ ]:
import math

card = tf.data.experimental.cardinality(train_dataset).numpy()  # number of batches
train_small = train_dataset.take(math.ceil(card * 0.2))

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="dog_breed_model.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]
history = model.fit(
    train_small,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=callbacks,
)

In [ ]:
import keras  # required if running this cell alone (e.g. after restart)
# test_dataset must exist: run the "Load datasets" cell first
test_model = keras.models.load_model("dog_breed_model.keras")
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

In [ ]:
import numpy as np
import keras
from sklearn.metrics import classification_report, confusion_matrix

# Requires: run "Load datasets" + "Build model" + "Train" cells first so model and test_dataset exist
y_true = np.concatenate([y.numpy() for _, y in test_dataset], axis=0)
y_prob = model.predict(test_dataset, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=test_dataset.class_names, digits=3))
